# ETL (Extract, Transform, Load) Revenue Sharing 00570
>An ETL proccess to retrieve data from estimated calculate income per store to send it to api endpoint landlord for jilid 00570 - DISTRICT 2 MEIKARTA

Created at : 2026-09-24

Created by lahiardhan@jiwagroup.com

# IMPORT LIBRARY

In [1]:
from datetime import datetime, timedelta
import mysql.connector
from mysql.connector import Error
from mysql.connector import errorcode
import time

from pytz import timezone

import pandas as pd
import numpy as np

import pymysql

import importlib, importlib.util

import warnings
warnings.filterwarnings("ignore")

from sqlalchemy import create_engine
import paramiko

import requests
import json

In [2]:
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

# CONFIG DB

In [3]:
# Import Config Data
import importlib, importlib.util

def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dw = module_from_file("config_dw", "../../db_config/config_db_datawarehouse.py")

# MAIN PROGRESS

### 1. Get Tokenization

In [4]:
import requests
import pandas as pd


def access_token(url, username, password):

    body = {
        'grant_type' : 'password',
        'username' : username,
        'password' : password 
    }

    response = requests.post(url, body)

    if response.status_code == 200:
        token_data = response.json()
        access_token = token_data.get("access_token")
        print("Access token obtained (hidden)")
        return access_token
    else:
        print("Failed : ", response.text)
        return None


# df = pd.read_csv('get_token_stg.csv')
df = pd.read_csv('../get_token.csv')
df['jilid'] = df['jilid'].astype(str).str.zfill(5)
df = df[df['jilid'] == '00570'].reset_index(drop=True)

df['access_token'] = df.apply(lambda row: access_token(row['url'], row['email'], row['password']), axis=1)
df['jilid'] = df['jilid'].astype(str).str.zfill(5)
df[['jilid']]  # never display email / password / access_token columns


### Get Date

In [5]:
yesterday = datetime.today().date() - timedelta(days=1)

# start_date = today
# end_date = start_date - timedelta(days=1)

# start_date_str = yesterday.strftime('%Y-%m-%d')
# end_date_str = yesterday.strftime('%Y-%m-%d')

start_date_str = '2026-10-08'
end_date_str = '2026-10-13'

print("today     :", yesterday)
print("start_day :", start_date_str)
print("end_day   :", end_date_str)

today     : 2026-10-13
start_day : 2026-10-08
end_day   : 2026-10-13


### 2. Extract Data Sales

In [ ]:
def get_est_income(start_date_str, end_date_str):
    conn = mysql.connector.connect(host=config_dw.hostname,
                             port=config_dw.port,
                             user=config_dw.username,
                             password=config_dw.password,
                             db=config_dw.db_target)
    # table daily_est_income_per_store_for_landlord: satu baris per date + outlet + payment_type
    query = """
        SELECT date, outlet_code, payment_type,
               subtotal, tax_amount, total_sales, sales_type_commission
        FROM daily_est_income_per_store_for_landlord
        WHERE outlet_code = '00570'
            AND date BETWEEN '{0}' AND '{1}'
        """.format(start_date_str, end_date_str)

    df_sales = pd.read_sql(query,conn)
    conn.close()
    return df_sales 

est_income = get_est_income(start_date_str, end_date_str)
est_income

In [ ]:
# payment type resmi yang boleh dikirim ke landlord (label = PaymentType di payload)
SENDABLE_PAYMENT_TYPES = [
    'Cash', 'BRI', 'Go Food', 'Grab Food', 'Grab Dine Out', 'Shopee Food',
    'Tokopedia Go', 'Jiwa+', 'iSeller QRIS', 'Mandiri QRIS', 'QPON',
]

# payment type yang ada transaksinya tapi belum resmi -> tidak dikirim, cek dulu kalau muncul
unsent = est_income[~est_income['payment_type'].isin(SENDABLE_PAYMENT_TYPES) & (est_income['total_sales'] != 0)]
if not unsent.empty:
    print("WARNING payment type belum resmi, TIDAK dikirim ke landlord:")
    print(unsent.groupby('payment_type')['total_sales'].sum())

sales = est_income[est_income['payment_type'].isin(SENDABLE_PAYMENT_TYPES) & (est_income['total_sales'] != 0)].copy()
sales['jilid'] = sales['outlet_code'].astype(str)
sales['date'] = sales['date'].astype(str)
sales['est_income'] = sales['subtotal'] - sales['sales_type_commission']   # nominal landlord = subtotal - komisi

final_data = sales.groupby(
    ['jilid', 'date', 'payment_type']
).agg({
    'total_sales': 'sum',
    'tax_amount': 'sum',
    'est_income': 'sum',
}).reset_index()

final_data.columns = ['jilid', 'TransactionDate', 'PaymentType', 'TotalSales', 'TotalTax', 'Amount']
final_data['TransactionNumber'] = 'Janji_Jiwa_' + final_data['jilid'] + '_' + final_data['PaymentType'].astype(str) + '_' + final_data["TransactionDate"].astype(str).str.replace('-', '', regex=False) + '_' + pd.Timestamp.now().strftime("%H%M%S")
# Count sharing amount to landlord 
final_data['Amount'] = final_data['Amount'].round(4)
final_data['Remarks'] = 'Janji_Jiwa_' + final_data['jilid'] + '_' + final_data['PaymentType'].astype(str) + '_' + final_data["TransactionDate"].astype(str) + '_' + pd.Timestamp.now().strftime("%H%M%S")

final_data

In [17]:
# print(final_data['Amount'].sum())
# print(final_data['Amount'].sum() * 0.1)
# print(final_data['Sharing'].sum())

### 3. Transform and Load Data to API

In [8]:
merged = pd.merge(final_data, df, on='jilid')
merged['url'] = 'https://revenueapi.lippomalls.com:8443/api/revenue/v1/save'

all_log_summary = []
all_log_detail = []

MAX_ATTEMPTS = 2
TIMEOUT_SECONDS = 90
BACKOFF_SECONDS = 5

for (jilid, transaction_date), group in merged.groupby(['jilid', 'TransactionDate']):
    url = group['url'].iloc[0]
    token = group['access_token'].iloc[0]
    revenue_data = group[['TransactionNumber', 'TransactionDate', 'Amount', 'Remarks']].to_dict('records')

    request_payload = {"revenueDatas": revenue_data}

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }

    response = None
    request_error = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            response = requests.post(url, json=request_payload, headers=headers, timeout=TIMEOUT_SECONDS)
            break
        except (requests.exceptions.Timeout, 
                requests.exceptions.ConnectionError,
                requests.exceptions.RequestException) as e:
            request_error = e
            if attempt < MAX_ATTEMPTS:
                print(f"RETRY | jilid: {jilid} | {transaction_date} | attempt {attempt} failed, waiting {BACKOFF_SECONDS}s: {str(e)}")
                time.sleep(BACKOFF_SECONDS)
            else:
                print(f"RETRY | jilid: {jilid} | {transaction_date} | attempt {attempt} failed: {str(e)}")

    now = datetime.now()

    if response is None:
        error_msg = f"Timeout/ConnectionError after {MAX_ATTEMPTS} attempts: {str(request_error)}"
        log_summary = pd.DataFrame([{
            'revenue_batch_id': None,
            'jilid': jilid,
            'transaction_date': transaction_date,
            'total_records_sent': len(revenue_data),
            'total_success': 0,
            'total_error': len(revenue_data),
            'http_status_code': None,
            'api_code': None,
            'api_message': error_msg,
            'api_url': url,
            'request_payload': json.dumps(request_payload),
            'response_payload': error_msg,
            'created_at': now
        }])
        all_log_summary.append(log_summary)

        detail = group[['jilid', 'TransactionNumber', 'TransactionDate', 'PaymentType', 'TotalSales', 'TotalTax', 'Amount', 'Remarks']].copy()
        detail.columns = ['jilid', 'transaction_number', 'transaction_date', 'payment_type', 'total_sales', 'total_tax', 'amount', 'remarks']
        detail['mall_transaction_id'] = None
        detail['revenue_batch_id'] = None
        detail['status'] = 'timeout_error'
        detail['error_message'] = error_msg
        detail['created_at'] = now
        all_log_detail.append(detail)

        print(f"TIMEOUT | jilid: {jilid} | {transaction_date} | skipped after {MAX_ATTEMPTS} attempts: {str(request_error)}")
        continue

    if response.status_code == 200:
        try:
            resp_json = response.json()
        except ValueError:
            resp_json = {
                'Message': f'Non-JSON 200 body: {response.text[:500]}',
                'transactionReturnDatas': [],
                'ErrorTransactionNumber': [],
            }

        log_summary = pd.DataFrame([{
            'revenue_batch_id': resp_json.get('RevenueBatchId'),
            'jilid': jilid,
            'transaction_date': transaction_date,
            'total_records_sent': len(revenue_data),
            'total_success': resp_json.get('TotalSuccess'),
            'total_error': resp_json.get('TotalError'),
            'http_status_code': response.status_code,
            'api_code': resp_json.get('Code'),
            'api_message': resp_json.get('Message'),
            'api_url': url,
            'request_payload': json.dumps(request_payload),
            'response_payload': json.dumps(resp_json),
            'created_at': now
        }])
        all_log_summary.append(log_summary)

        request_df = group[['jilid', 'TransactionNumber', 'TransactionDate', 'PaymentType', 'TotalSales', 'TotalTax', 'Amount', 'Remarks']].copy()
        request_df.columns = ['jilid', 'transaction_number', 'transaction_date', 'payment_type', 'total_sales', 'total_tax', 'amount', 'remarks']

        response_details = pd.DataFrame(resp_json.get('transactionReturnDatas', []))
        if not response_details.empty:
            response_details = response_details.rename(columns={
                'TenantTransactionNumber': 'transaction_number',
                'MallTransactionId': 'mall_transaction_id'
            })
            detail = request_df.merge(response_details, on='transaction_number', how='left')
        else:
            detail = request_df.copy()
            detail['mall_transaction_id'] = None

        error_txns = resp_json.get('ErrorTransactionNumber', [])
        error_map = {}
        for err in error_txns:
            if '|' in str(err):
                txn_num, error_msg = str(err).split('|', 1)
                error_map[txn_num] = error_msg
            else:
                error_map[str(err)] = 'unknown error'

        detail['revenue_batch_id'] = resp_json.get('RevenueBatchId')
        detail['status'] = detail['mall_transaction_id'].apply(lambda x: 'success' if pd.notna(x) else 'error')
        detail['error_message'] = detail['transaction_number'].map(error_map)
        detail['created_at'] = now
        all_log_detail.append(detail)

        print(f"SUCCESS | jilid: {jilid} | {transaction_date} | {response.status_code} | {resp_json.get('Message')}")
    else:
        log_summary = pd.DataFrame([{
            'revenue_batch_id': None,
            'jilid': jilid,
            'transaction_date': transaction_date,
            'total_records_sent': len(revenue_data),
            'total_success': 0,
            'total_error': len(revenue_data),
            'http_status_code': response.status_code,
            'api_code': None,
            'api_message': response.text[:500],
            'api_url': url,
            'request_payload': json.dumps(request_payload),
            'response_payload': response.text[:500],
            'created_at': now
        }])
        all_log_summary.append(log_summary)

        detail = group[['jilid', 'TransactionNumber', 'TransactionDate', 'PaymentType', 'TotalSales', 'TotalTax', 'Amount', 'Remarks']].copy()
        detail.columns = ['jilid', 'transaction_number', 'transaction_date', 'payment_type', 'total_sales', 'total_tax','amount', 'remarks']
        detail['mall_transaction_id'] = None
        detail['revenue_batch_id'] = None
        detail['status'] = 'http_error'
        detail['error_message'] = f"HTTP {response.status_code}: {response.text[:200]}"
        detail['created_at'] = now
        all_log_detail.append(detail)

        print(f"FAILED | jilid: {jilid} | {transaction_date} | {response.status_code} | {response.text[:100]}")

df_log_summary = pd.concat(all_log_summary, ignore_index=True) if all_log_summary else pd.DataFrame()
df_log_detail = pd.concat(all_log_detail, ignore_index=True) if all_log_detail else pd.DataFrame()

print(f"\nTotal log summary: {len(df_log_summary)} rows")
print(f"Total log detail: {len(df_log_detail)} rows")

SUCCESS | jilid: 00570 | 2026-10-08 | 200 | Success
SUCCESS | jilid: 00570 | 2026-10-09 | 200 | Success
SUCCESS | jilid: 00570 | 2026-10-10 | 200 | Success
SUCCESS | jilid: 00570 | 2026-10-11 | 200 | Success
SUCCESS | jilid: 00570 | 2026-10-12 | 200 | Success
SUCCESS | jilid: 00570 | 2026-10-13 | 200 | Success

Total log summary: 6 rows
Total log detail: 49 rows


In [9]:
df_log_summary.head(3)

,revenue_batch_id,jilid,transaction_date,total_records_sent,total_success,total_error,http_status_code,api_code,api_message,api_url,request_payload,response_payload,created_at
0,1501126,00570,2026-10-08,8,8,0,200,0,Success,https://revenueapi.lippomalls.com:8443/api/rev...,"{""revenueDatas"": [{""TransactionNumber"": ""Janji...","{""RevenueBatchId"": 1501126, ""TotalSuccess"": 8,...",2026-10-14 10:56:03.307308
1,1501127,00570,2026-10-09,9,9,0,200,0,Success,https://revenueapi.lippomalls.com:8443/api/rev...,"{""revenueDatas"": [{""TransactionNumber"": ""Janji...","{""RevenueBatchId"": 1501127, ""TotalSuccess"": 9,...",2026-10-14 10:56:03.740912
2,1501128,00570,2026-10-10,8,8,0,200,0,Success,https://revenueapi.lippomalls.com:8443/api/rev...,"{""revenueDatas"": [{""TransactionNumber"": ""Janji...","{""RevenueBatchId"": 1501128, ""TotalSuccess"": 8,...",2026-10-14 10:56:06.862343


In [10]:
df_log_detail.head(3)

,jilid,transaction_number,transaction_date,payment_type,total_sales,total_tax,amount,remarks,mall_transaction_id,revenue_batch_id,status,error_message,created_at
0,00570,Janji_Jiwa_00570_Cash_20261008_105551,2026-10-08,Cash,57000.0000,5181.8182,51818.1818,Janji_Jiwa_00570_Cash_2026-10-08_105551,71694511,1501126,success,NaN,2026-10-14 10:56:03.307308
1,00570,Janji_Jiwa_00570_Go Food_20261008_105551,2026-10-08,Go Food,369080.0000,33753.3825,281674.3575,Janji_Jiwa_00570_Go Food_2026-10-08_105551,71694512,1501126,success,NaN,2026-10-14 10:56:03.307308
2,00570,Janji_Jiwa_00570_Grab Food_20261008_105551,2026-10-08,Grab Food,685500.0000,62690.8272,529387.7928,Janji_Jiwa_00570_Grab Food_2026-10-08_105551,71694513,1501126,success,NaN,2026-10-14 10:56:03.307308


In [11]:
load_data_by = 'lahiardhan@jiwagroup.com'

df_log_summary['load_data_by'] = load_data_by
df_log_detail['load_data_by'] = load_data_by

In [12]:
df_cekkk = df_log_detail.groupby(['transaction_date']).agg({'amount': 'sum'}).reset_index()

df_cekkk

,transaction_date,amount
0,2026-10-08,3019441.2965
1,2026-10-09,3356005.2543
2,2026-10-10,3270176.7877
3,2026-10-11,2399103.9213
4,2026-10-12,1960268.7562
5,2026-10-13,2135452.8457


### 4. Dump Log to DB (api_landlord_log + api_landlord_log_detail)

In [13]:
engine = create_engine("mysql+pymysql://{user}:{pw}@{hostsname}/{db}".format(
    hostsname=config_dw.hostname,
    user=config_dw.username,
    pw=config_dw.password,
    db=config_dw.db_resource
))

In [14]:
if not df_log_summary.empty:
    df_log_summary.to_sql('api_landlord_logs', con=engine, if_exists='append', index=False)
    print(f"api_landlord_log: {len(df_log_summary)} rows inserted")
else:
    print("api_landlord_log: no data to insert")

api_landlord_log: 6 rows inserted


In [15]:
df_log_detail

,jilid,transaction_number,transaction_date,payment_type,total_sales,total_tax,amount,remarks,mall_transaction_id,revenue_batch_id,status,error_message,created_at,load_data_by
0,00570,Janji_Jiwa_00570_Cash_20261008_105551,2026-10-08,Cash,57000.0000,5181.8182,51818.1818,Janji_Jiwa_00570_Cash_2026-10-08_105551,71694511,1501126,success,NaN,2026-10-14 10:56:03.307308,lahiardhan@jiwagroup.com
1,00570,Janji_Jiwa_00570_Go Food_20261008_105551,2026-10-08,Go Food,369080.0000,33753.3825,281674.3575,Janji_Jiwa_00570_Go Food_2026-10-08_105551,71694512,1501126,success,NaN,2026-10-14 10:56:03.307308,lahiardhan@jiwagroup.com
2,00570,Janji_Jiwa_00570_Grab Food_20261008_105551,2026-10-08,Grab Food,685500.0000,62690.8272,529387.7928,Janji_Jiwa_00570_Grab Food_2026-10-08_105551,71694513,1501126,success,NaN,2026-10-14 10:56:03.307308,lahiardhan@jiwagroup.com
3,00570,Janji_Jiwa_00570_Jiwa+_20261008_105551,2026-10-08,Jiwa+,250716.0000,22821.0287,227894.9713,Janji_Jiwa_00570_Jiwa+_2026-10-08_105551,71694514,1501126,success,NaN,2026-10-14 10:56:03.307308,lahiardhan@jiwagroup.com
4,00570,Janji_Jiwa_00570_QPON_20261008_105551,2026-10-08,QPON,59400.0000,5400.0000,51300.0000,Janji_Jiwa_00570_QPON_2026-10-08_105551,71694515,1501126,success,NaN,2026-10-14 10:56:03.307308,lahiardhan@jiwagroup.com
5,00570,Janji_Jiwa_00570_Shopee Food_20261008_105551,2026-10-08,Shopee Food,896472.6000,81783.2006,692485.9894,Janji_Jiwa_00570_Shopee Food_2026-10-08_105551,71694516,1501126,success,NaN,2026-10-14 10:56:03.307308,lahiardhan@jiwagroup.com
6,00570,Janji_Jiwa_00570_Tokopedia Go_20261008_105551,2026-10-08,Tokopedia Go,371100.0000,33736.3636,313748.1864,Janji_Jiwa_00570_Tokopedia Go_2026-10-08_105551,71694517,1501126,success,NaN,2026-10-14 10:56:03.307308,lahiardhan@jiwagroup.com
7,00570,Janji_Jiwa_00570_iSeller QRIS_20261008_105551,2026-10-08,iSeller QRIS,965000.0000,87727.2727,871131.8173,Janji_Jiwa_00570_iSeller QRIS_2026-10-08_105551,71694518,1501126,success,NaN,2026-10-14 10:56:03.307308,lahiardhan@jiwagroup.com
8,00570,Janji_Jiwa_00570_BRI_20261009_105551,2026-10-09,BRI,21000.0000,1909.0909,19090.9091,Janji_Jiwa_00570_BRI_2026-10-09_105551,71694519,1501127,success,NaN,2026-10-14 10:56:03.740912,lahiardhan@jiwagroup.com
9,00570,Janji_Jiwa_00570_Cash_20261009_105551,2026-10-09,Cash,142000.0000,12909.0909,129090.9091,Janji_Jiwa_00570_Cash_2026-10-09_105551,71694520,1501127,success,NaN,2026-10-14 10:56:03.740912,lahiardhan@jiwagroup.com


In [16]:
if not df_log_detail.empty:
    df_log_detail.to_sql('api_landlord_log_detail', con=engine, if_exists='append', index=False)
    print(f"api_landlord_log_detail: {len(df_log_detail)} rows inserted")
else:
    print("api_landlord_log_detail: no data to insert")

api_landlord_log_detail: 49 rows inserted


### 5. Send Email to Data Team

> Cells below are for sending notification email

In [17]:
import os
import smtplib, ssl
from dotenv import load_dotenv

load_dotenv()  # reads GMAIL_SMTP_PASSWORD from .env (never hardcode / commit it)

port = 587  # For starttls
smtp_server = "smtp.gmail.com"
sender_email = "saputra.christabel20@gmail.com"
receiver_email = ["athens.jiwagroup@gmail.com", "lahia.ardhanlahia@gmail.com"]
password = os.getenv("GMAIL_SMTP_PASSWORD")
if not password:
    raise RuntimeError("GMAIL_SMTP_PASSWORD is not set in the environment / .env")

message = """Subject: [NEW] - 00570 - Send API Landlord Yesterday - Done

Hi ! This is reporting Data for You !

This message is sent from Jupyter Notebook.

Best Regards,
DatO ( Data autOmation )"""

context = ssl.create_default_context()
with smtplib.SMTP(smtp_server, port) as server:
    server.ehlo()  # Can be omitted
    server.starttls(context=context)
    server.ehlo()  # Can be omitted
    server.login(sender_email, password)
    server.sendmail(sender_email, receiver_email, message)